# Smart Meter Tamper Labeling — Rule-Based (IS15959-style thresholds)

Labels every timestamp in the dataset with applicable tamper type(s), based on the
occurrence thresholds from `Tamper_Thresholds_(10-60A).doc`.

## Constants used (from the document)
- `Vref` = 240 V
- `Ibasic` = 10 A
- `Imax` = **40 A** (changed from the document's 60 A, per instruction)

## Important scope notes — read before using

**Not computable from this dataset** (require sensors/internal meter state not present
in `MID, Load Survey Time, R/Y/B Current, R/Y/B Voltage, Active/Apparent Energy,
Reactive Lag/Lead, PF`):
- **Magnetic Tamper** — needs a magnetic-field sensor flag
- **Top Cover Open** — needs a physical cover-open switch signal
- **Two Wire** — marked `NA` in the source document itself

These columns will still appear in the output, always `0`/`False`, with a note — they are
**not silently dropped**, so you can see exactly what wasn't evaluated.

**Computable, with approximations:**
- **Very Low PF** — uses overall `PF` in place of per-phase Net PF (dataset has one PF
  column, not phase-wise).
- **CT Reversal** — the dataset's `PF` column contains **negative values, confirmed to
  encode energy-flow direction** (negative = reverse/export), used as the `Direction`
  term: `Direction -ve` → `PF < 0`. Magnitude terms use overall `PF`, same approximation
  as Very Low PF.
- **CT Bypass** — `Ibypass` confirmed to be the meter's own measured phase current
  (not a separate bypass-path sensor), so it maps directly to `Ir`/`Iy`/`Ib`. `Iavg` is
  the average current across R/Y/B (already computed as `_Iavg_val`).
- **CT Open (phase-wise)** — `Ix` maps directly to `Ir`/`Iy`/`Ib`. `I Primary` has no
  direct equivalent in the dataset, so it's **approximated** using the other two
  phases' currents (if either is meaningfully above threshold, real current is assumed
  to be flowing through the load). This assumes a roughly balanced 3-phase load and
  will under-detect CT Open on a phase that's normally lightly loaded relative to the
  others — worth watching for in the results.

**Phase reporting:** phase-wise rules (Missing Potential, High Voltage, Low Voltage,
Over Current, CT Reversal, CT Bypass, CT Open) are evaluated per phase internally but
**reported as a single collapsed label** — the column is `1` if the condition fires on
ANY of R/Y/B, with no phase noted. If you need to know which specific phase triggered,
that requires re-splitting these back into per-phase columns.

**Labeling method:** occurrence-threshold check per timestamp (a snapshot classification),
not the full occurrence→persistence→restoration state machine described in the document
(which needs multi-row event duration tracking — 30s/5min persistence — and is a
reasonable follow-up if you want event-level tamper detection rather than per-timestamp
flags).

**Missing Value handling:** if every sensor column in a row is NaN, the row is labeled
`"Missing Value"` and no other tamper rule is evaluated for it.


In [1]:
import pandas as pd
import numpy as np

## Config — column names & constants

In [2]:
TIME_COL = "Load Survey Time"
ID_COL   = "MID"

CURRENT_COLS = ["R Phase Current", "Y Phase Current", "B Phase Current"]
VOLTAGE_COLS = ["R Phase Voltage", "Y Phase Voltage", "B Phase Voltage"]
ENERGY_COLS  = ["Active Energy", "Apparent Energy", "Reactive Lag", "Reactive Lead"]
PF_COL = "PF"

NUMERIC_COLS = CURRENT_COLS + VOLTAGE_COLS + ENERGY_COLS + [PF_COL]

# --- Constants from the tamper threshold document ---
VREF    = 240   # Volts
IBASIC  = 10    # Amps (Ib)
IMAX    = 40    # Amps — changed from the document's 60A per instruction

## Load data
Use either the raw dataset or the output of the earlier preprocessing notebook.

In [3]:
df = pd.read_csv("Dataset/cleaned_data.csv")
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df.head()

,MID,Load Survey Time,R Phase Current,Y Phase Current,B Phase Current,R Phase Voltage,Y Phase Voltage,B Phase Voltage,Active Energy,Apparent Energy,Reactive Lag,Reactive Lead,PF,time_diff
0,900069,2017-01-01 00:00:00,34.38,34.30,0.0,261.0,255.4,0.0,1.20,1.29,0.34,0.34,0.930,NaN
1,900069,2017-01-01 00:15:00,42.93,42.83,0.0,260.3,253.4,0.0,1.09,1.18,0.32,0.32,0.924,0 days 00:15:00
2,900069,2017-01-01 00:30:00,0.00,0.00,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.000,0 days 00:15:00
3,900069,2017-01-01 00:45:00,0.00,0.00,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.000,0 days 00:15:00
4,900069,2017-01-01 01:00:00,0.00,0.00,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.000,0 days 00:15:00


## Step 1 — Missing Value flag
Rows where every sensor column is NaN. These rows are excluded from all other tamper checks.

In [4]:
def flag_missing_value(df: pd.DataFrame) -> pd.Series:
    return df[NUMERIC_COLS].isna().all(axis=1)

missing_mask = flag_missing_value(df)
print(f"Missing-value rows: {missing_mask.sum()} / {len(df)}")

Missing-value rows: 31564 / 1707291


## Step 1b — Power Cut flag
Rows where every sensor column is present but **exactly 0** (current, voltage, energy, PF all 0) — the meter is alive and reporting, just seeing no power, unlike `Missing Value` where the row is all `NaN` (no reading at all). These rows are excluded from all other tamper checks, same as `Missing Value`.

In [5]:
def flag_power_cut(df: pd.DataFrame) -> pd.Series:
    """True where every numeric sensor column reads exactly 0 (not NaN) for the row."""
    return (df[NUMERIC_COLS] == 0).all(axis=1)

power_cut_mask = flag_power_cut(df) & ~missing_mask
print(f"Power-cut rows: {power_cut_mask.sum()} / {len(df)}")

Power-cut rows: 156316 / 1707291


## Step 2 — Derived per-row electrical quantities
Needed for the imbalance / avg-current style rules.

In [6]:
def add_derived_quantities(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["_Imax_val"] = df[CURRENT_COLS].max(axis=1)
    df["_Imin_val"] = df[CURRENT_COLS].min(axis=1)
    df["_Iavg_val"] = df[CURRENT_COLS].mean(axis=1)

    df["_Vmax_val"] = df[VOLTAGE_COLS].max(axis=1)
    df["_Vmin_val"] = df[VOLTAGE_COLS].min(axis=1)
    return df

df = add_derived_quantities(df)

## Step 3 — Non-computable tamper types (documented, always 0)
Only 3 remain — Magnetic Tamper, Top Cover Open, Two Wire. Kept as explicit columns so nothing is silently missing from the schema.

In [7]:
# NOT_COMPUTABLE_TYPES = [
#     "Magnetic Tamper",  # needs magnetic field sensor
#     "Top Cover Open",   # needs physical cover-open switch
#     "Two Wire",         # marked NA in source document
# ]

# for t in NOT_COMPUTABLE_TYPES:
#     df[t] = 0  # always 0 — not evaluable from available sensor columns

## Step 4 — Computable tamper rules (vectorized, occurrence thresholds)

Each rule below mirrors the "Occurrence Thresholds" column of the document exactly,
substituted with `Vref=240`, `Ibasic=10`, `Imax=40`. Includes CT Reversal (using `PF`
sign as direction proxy), CT Bypass, and CT Open (see approximation notes above).

In [8]:
Vr, Vy, Vb = df[VOLTAGE_COLS[0]], df[VOLTAGE_COLS[1]], df[VOLTAGE_COLS[2]]
Ir, Iy, Ib = df[CURRENT_COLS[0]], df[CURRENT_COLS[1]], df[CURRENT_COLS[2]]

all_V_gt_60pct  = (Vr > 0.60*VREF) & (Vy > 0.60*VREF) & (Vb > 0.60*VREF)
all_V_lt_115pct = (Vr < 1.15*VREF) & (Vy < 1.15*VREF) & (Vb < 1.15*VREF)
all_V_lt_125pct = (Vr < 1.25*VREF) & (Vy < 1.25*VREF) & (Vb < 1.25*VREF)
all_I_gt_10pctIb = (Ir > 0.10*IBASIC) & (Iy > 0.10*IBASIC) & (Ib > 0.10*IBASIC)

# --- 7. Current Imbalance ---
df["Current Imbalance"] = (
    ((df["_Imax_val"] - df["_Imin_val"]) > 0.20 * df["_Iavg_val"]) &
    (df["_Iavg_val"] > 0.10 * IBASIC) &
    all_I_gt_10pctIb &
    all_V_gt_60pct
).astype(int)

# --- 8. Voltage Unbalance ---
df["Voltage Unbalance"] = (
    ((df["_Vmax_val"] - df["_Vmin_val"]) > 0.20 * df["_Vmax_val"]) &
    all_V_gt_60pct &
    all_V_lt_115pct
).astype(int)

# --- 9. Missing Potential ---
# Phase-wise by definition in the document, but collapsed here to a single label per
# timestamp: flagged 1 if the condition fires on ANY of R/Y/B.
def missing_potential(Vx, Ix, others):
    other_ok = pd.concat(others, axis=1).gt(0.60*VREF).any(axis=1)
    return (Vx < 0.20*VREF) & (Ix > 0.10*IBASIC) & other_ok

_missing_potential_any = (
    missing_potential(Vr, Ir, [Vy, Vb]) |
    missing_potential(Vy, Iy, [Vr, Vb]) |
    missing_potential(Vb, Ib, [Vr, Vy])
)
df["Missing Potential"] = _missing_potential_any.astype(int)

# --- 10. High Voltage ---
_high_voltage_any = (
    ((Vr > 1.15*VREF) & all_V_lt_125pct) |
    ((Vy > 1.15*VREF) & all_V_lt_125pct) |
    ((Vb > 1.15*VREF) & all_V_lt_125pct)
)
df["High Voltage"] = _high_voltage_any.astype(int)

# --- 11. Low Voltage ---
def low_voltage(Vx):
    return ((Vx > 0.20*VREF) & (Vx < 0.75*VREF) & all_V_lt_125pct)

_low_voltage_any = low_voltage(Vr) | low_voltage(Vy) | low_voltage(Vb)
df["Low Voltage"] = _low_voltage_any.astype(int)

# --- 12. Over Current ---
# Note: document also excludes "magnet" cases — not evaluable here (see Magnetic Tamper above)
_over_current_any = (
    ((Ir > 1.20*IMAX) & (Vr > 0.60*VREF)) |
    ((Iy > 1.20*IMAX) & (Vy > 0.60*VREF)) |
    ((Ib > 1.20*IMAX) & (Vb > 0.60*VREF))
)
df["Over Current"] = _over_current_any.astype(int)

# --- 13. Very Low PF (approximated with overall PF — see notes above) ---
# Document uses per-phase notation "Vx>60%Vref and Ix>10%Ib, x=r/y/b", same pattern as
# Over Current — so this is OR'd across phases, not required on all three at once.
def very_low_pf(Vx, Ix):
    return (Vx > 0.60*VREF) & (Ix > 0.10*IBASIC)

df["Very Low PF"] = (
    (df[PF_COL] < 0.3) &
    (very_low_pf(Vr, Ir) | very_low_pf(Vy, Iy) | very_low_pf(Vb, Ib))
).astype(int)

# --- 14. Neutral Disturbance ---
df["Neutral Disturbance"] = (
    (Vr > 1.25*VREF) | (Vy > 1.25*VREF) | (Vb > 1.25*VREF)
).astype(int)

# --- 4b. CT Reversal ---
# Direction proxy: PF sign (confirmed negative values present in this dataset, encoding
# reverse/export flow). Magnitude terms use overall PF as a stand-in for per-phase Net PF
# (same approximation as Very Low PF — see notes above).
def ct_reversal(Vx, Ix):
    return (Vx > 0.60*VREF) & (Ix > 0.05*IBASIC) & (df[PF_COL].abs() > 0.2) & (df[PF_COL] < 0)

_ct_reversal_any = ct_reversal(Vr, Ir) | ct_reversal(Vy, Iy) | ct_reversal(Vb, Ib)
df["CT Reversal"] = _ct_reversal_any.astype(int)

# --- 4c. CT Bypass ---
# Ibypass = the meter's own measured current at that phase (confirmed — not a separate
# bypass-path sensor reading). Iavg = average current across R/Y/B (already computed).
def ct_bypass(Ibypass_x):
    return (Ibypass_x > 0.30*IBASIC) & (df["_Iavg_val"] > 0.10*IBASIC)

_ct_bypass_any = ct_bypass(Ir) | ct_bypass(Iy) | ct_bypass(Ib)
df["CT Bypass"] = _ct_bypass_any.astype(int)

# --- 4d. CT Open ---
# Ix = the phase's own measured current (already used elsewhere as Ir/Iy/Ib).
# I Primary approximated using the OTHER two phases' currents as a reference for
# "is real current actually flowing" — same pattern as the Missing Potential rule's
# "any other phase voltage" check. ASSUMPTION: relies on roughly balanced 3-phase load;
# will under-detect CT Open on a phase that is normally lightly loaded relative to others.
def ct_open(Vx, Ix, others):
    other_current_high = pd.concat(others, axis=1).max(axis=1) > 0.30*IBASIC
    return (Vx > 0.60*VREF) & (Ix < 0.01*IBASIC) & other_current_high

_ct_open_any = ct_open(Vr, Ir, [Iy, Ib]) | ct_open(Vy, Iy, [Ir, Ib]) | ct_open(Vb, Ib, [Ir, Iy])
df["CT Open"] = _ct_open_any.astype(int)

## Step 5 — Apply Missing Value / Power Cut overrides & build final multi-label column

For rows flagged as `Missing Value` (all NaN) or `Power Cut` (all exactly 0), all other tamper flags are forced to 0 — those rows can't meaningfully trigger a tamper rule, so they get a single dedicated label instead. Priority when building the final label: `Missing Value` > `Power Cut` > computed tamper types > `Normal`.

In [9]:
COMPUTED_TAMPER_COLS = [
    "Current Imbalance", "Voltage Unbalance",
    "Missing Potential",
    "High Voltage",
    "Low Voltage",
    "Over Current",
    "Very Low PF", "Neutral Disturbance",
    "CT Reversal",
    "CT Bypass",
    "CT Open",
]
ALL_TAMPER_COLS = COMPUTED_TAMPER_COLS

df["Missing Value"] = missing_mask.astype(int)
df["Power Cut"] = power_cut_mask.astype(int)
df.loc[missing_mask, ALL_TAMPER_COLS] = 0
df.loc[power_cut_mask, ALL_TAMPER_COLS] = 0

def build_label_list(row):
    if row["Missing Value"] == 1:
        return ["Missing Value"]
    if row["Power Cut"] == 1:
        return ["Power Cut"]
    labels = [t for t in COMPUTED_TAMPER_COLS if row[t] == 1]
    return labels if labels else ["Normal"]

df["Tamper_Labels"] = df.apply(build_label_list, axis=1)

df[["Missing Value", "Power Cut"] + COMPUTED_TAMPER_COLS + ["Tamper_Labels"]].head(10)

,Missing Value,Power Cut,Current Imbalance,Voltage Unbalance,Missing Potential,High Voltage,Low Voltage,Over Current,Very Low PF,Neutral Disturbance,CT Reversal,CT Bypass,CT Open,Tamper_Labels
0,0,0,0,0,0,0,0,0,0,0,0,1,0,[CT Bypass]
1,0,0,0,0,0,0,0,0,0,0,0,1,0,[CT Bypass]
2,0,1,0,0,0,0,0,0,0,0,0,0,0,[Power Cut]
3,0,1,0,0,0,0,0,0,0,0,0,0,0,[Power Cut]
4,0,1,0,0,0,0,0,0,0,0,0,0,0,[Power Cut]
5,0,0,0,0,0,0,0,0,0,0,0,1,0,[CT Bypass]
6,0,0,0,0,0,0,0,0,0,0,0,1,0,[CT Bypass]
7,0,0,0,0,0,0,0,0,0,0,0,1,0,[CT Bypass]
8,0,0,0,0,0,0,0,0,0,0,0,1,0,[CT Bypass]
9,0,0,0,0,0,0,0,0,0,0,0,1,0,[CT Bypass]


## Step 6 — Label distribution (sanity check)

In [10]:
label_counts = pd.Series(
    [lbl for labels in df["Tamper_Labels"] for lbl in labels]
).value_counts()
label_counts

CT Bypass              918140
Current Imbalance      490547
CT Reversal            450994
Very Low PF            412995
Normal                 412637
Power Cut              156316
CT Open                125123
Missing Value           31564
Neutral Disturbance      8634
Low Voltage              6817
Over Current             1878
Voltage Unbalance         607
High Voltage              306
Missing Potential         212
Name: count, dtype: int64

## Step 7 — Save labeled dataset

In [11]:
df = df.drop(columns=["_Imax_val", "_Imin_val", "_Iavg_val", "_Vmax_val", "_Vmin_val"])
df.to_csv("clean_data_labeled.csv", index=False)
print("Saved clean_data_labeled.csv —", df.shape)

Saved clean_data_labeled.csv — (1707291, 28)
